# Use the WLCF Emulator

Run `emulator.ipynb` first. This notebook loads the generated emulator, demonstrates predictions, fits the test-set point closest to a Planck-like cosmology with an artificial covariance, and makes a triangular MCMC plot.

## Setup

In [ ]:
%matplotlib inline

from pathlib import Path
import importlib.util
import sys

import numpy as np
import matplotlib.pyplot as plt
import corner


def find_tests_dir(start=Path.cwd()):
    for path in [start, *start.parents]:
        if (path / "emulator.py").exists():
            return path
        if (path / "tests" / "emulator.py").exists():
            return path / "tests"
    raise FileNotFoundError("Could not find tests/emulator.py. Open this notebook from the repository or tests folder.")


TEST_DIR = find_tests_dir()
MODULE_PATH = TEST_DIR / "emulator.py"
spec = importlib.util.spec_from_file_location("emulator", MODULE_PATH)
flow = importlib.util.module_from_spec(spec)
sys.modules["emulator"] = flow
spec.loader.exec_module(flow)

plt.rcParams.update({
    "figure.dpi": 120,
    "savefig.dpi": 180,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "font.size": 11,
})

config = flow.EmulatorConfig()
paths = flow.make_paths(config)

required = [paths.weights_path, paths.grid_path, paths.vector_dir]
missing = [item for item in required if not item.exists()]
if missing:
    missing_text = "\n".join(str(item) for item in missing)
    raise FileNotFoundError(
        "Missing emulator files. Run emulator.ipynb first.\n\n"
        + missing_text
    )

print(f"Weights: {paths.weights_path}")
print(f"Grid:    {paths.grid_path}")
print(f"Vectors: {paths.vector_dir}")
print(f"Output:  {paths.usage_output_dir}")
print(f"Loaded module: {MODULE_PATH}")

## Load the Emulator

In [ ]:
emulator = flow.WLCFEmulator(paths.weights_path)

print("Parameters:", emulator.param_names)
print("Bounds:")
for name, bounds in zip(emulator.param_names, emulator.bounds):
    print(f"  {name:8s}: {bounds[0]:g} to {bounds[1]:g}")
print("Moments:", emulator.moments)
print("Target representation:", emulator.target_representation)
print("Target shape:", emulator.target_shape)

## Basic Emulator Prediction

In [ ]:
example_params = emulator.center_parameters()
example_moments = emulator.moments[:3]
example_vector = emulator.predict_vector(example_params, moments=example_moments)

print("Example parameters:", example_params)
print("Moments:", example_moments)
print("Prediction length:", example_vector.size)
print("Prediction finite:", np.isfinite(example_vector).all())

In [ ]:
pieces = emulator.split_vector_by_moment(example_vector, moments=example_moments)
fig, axes = plt.subplots(len(example_moments), 1, figsize=(8.5, 2.4 * len(example_moments)), constrained_layout=True)
axes = np.atleast_1d(axes)
for ax, moment in zip(axes, example_moments):
    ax.plot(pieces[moment], lw=1.0)
    ax.set_yscale("symlog", linthresh=1.0e-12)
    ax.set_ylabel(rf"$\zeta_{{{moment}}}$")
    ax.set_xlabel("native vector index")
fig.suptitle("Example emulator prediction", y=1.02)
example_plot = paths.usage_output_dir / "example_emulator_prediction.png"
fig.savefig(example_plot, bbox_inches="tight")
print(f"Saved {example_plot}")

## Choose a Planck-Like Test-Set Point

The helper reconstructs the same train/validation/test split used in notebook 1 and selects the held-out test-set point closest to a Planck-like reference cosmology. This keeps the example close to familiar cosmological values while still fitting a point not used for training.

In [ ]:
planck_like_params = flow.PLANCK_2018_REFERENCE
rng = np.random.default_rng(config.random_test_seed - 1)
grid, X_test, sample_id, truth, target_path = flow.choose_test_sample(
    config,
    paths,
    emulator,
    rng=rng,
    target_params=planck_like_params,
)

target_vector = flow.load_target_vector(target_path, emulator)
planck_theta = emulator.theta_array(planck_like_params)
normalized_delta = (truth - planck_theta) / (emulator.bounds[:, 1] - emulator.bounds[:, 0])

print(f"Generated vectors: {len(grid)}")
print(f"Selected Planck-like test sample: {sample_id:04d}")
print("Planck-like reference:")
for name, value in zip(emulator.param_names, planck_theta):
    print(f"  {name:8s} = {value:.8g}")
print("Truth for selected test sample:")
for name, value, delta in zip(emulator.param_names, truth, truth - planck_theta):
    print(f"  {name:8s} = {value:.8g}  delta_from_reference={delta:+.3e}")
print(f"Normalized distance to reference: {np.linalg.norm(normalized_delta):.4f}")
print(f"Target file: {target_path}")
print(f"Full target length: {target_vector.size}")

## Artificial Covariance and Selected Data Vector

The raw emulator output is large, so the demo fits a reproducible subset of entries from every multipole. The selected indices and covariance are saved.

In [ ]:
# Broad demo covariance so the true value is inside the triangle-plot contours.
# Replace this with the real covariance for a scientific fit.
config.mcmc_fractional_error = 1.

indices_path = paths.usage_output_dir / f"selected_indices_sample_{sample_id:04d}.npy"
selected_indices = flow.build_selected_indices(config, emulator, output_path=indices_path)
data_vector = target_vector[selected_indices]

variance_path = paths.usage_output_dir / f"artificial_variance_sample_{sample_id:04d}.npy"
covariance_path = paths.usage_output_dir / f"artificial_covariance_sample_{sample_id:04d}.npy"
sigma, variance, covariance = flow.build_artificial_covariance(
    data_vector,
    config,
    variance_path=variance_path,
    covariance_path=covariance_path,
)

print("Selected data-vector length:", data_vector.size)
print("Artificial covariance shape:", covariance.shape)
print("Fractional error:", config.mcmc_fractional_error)
print(f"Saved {indices_path}")
print(f"Saved {variance_path}")
print(f"Saved {covariance_path}")

## MCMC Fit with `emcee`

In [ ]:
predict_selected = flow.make_selected_predictor(emulator, selected_indices)
check = predict_selected(truth)
print("Selected prediction length:", check.size)
print("Selected prediction finite:", np.isfinite(check).all())

fit = flow.run_mcmc_fit(
    emulator,
    predict_selected,
    data_vector,
    sigma,
    truth,
    config,
    rng=rng,
)

chain = fit["chain"]
raw_chain = fit["raw_chain"]
chain_path = paths.usage_output_dir / f"mcmc_chain_sample_{sample_id:04d}.npy"
raw_chain_path = paths.usage_output_dir / f"mcmc_raw_chain_sample_{sample_id:04d}.npy"
np.save(chain_path, chain)
np.save(raw_chain_path, raw_chain)

print("Least-squares best fit:")
for name, truth_i, value in zip(emulator.param_names, truth, fit["best_theta"]):
    print(f"  {name:8s} = {value:.8g}  truth={truth_i:.8g}  delta={value - truth_i:+.3e}")
print("Mean acceptance fraction:", fit["acceptance_fraction"])
print("Samples after burn-in:", len(chain))
print(f"Saved {chain_path}")

## Posterior Summary

In [ ]:
summary = flow.summarize_chain(chain, emulator.param_names, truth=truth)
summary_path = paths.usage_output_dir / f"mcmc_summary_sample_{sample_id:04d}.csv"
summary.to_csv(summary_path, index=False)
print(f"Saved {summary_path}")
summary

## Triangular Plot

In [ ]:
corner_fig = corner.corner(
    chain,
    labels=emulator.param_names,
    show_titles=True,
    title_fmt=".5f",
    quantiles=[0.16, 0.5, 0.84],
)

# Draw the true parameter values explicitly in blue, above the corner contours.
ndim = len(emulator.param_names)
axes = np.asarray(corner_fig.axes).reshape((ndim, ndim))
truth_color = "tab:blue"

for i in range(ndim):
    axes[i, i].axvline(truth[i], color=truth_color, lw=2.4, ls="-", zorder=20)
    for j in range(i):
        ax = axes[i, j]
        ax.axvline(truth[j], color=truth_color, lw=1.8, ls="-", alpha=0.9, zorder=20)
        ax.axhline(truth[i], color=truth_color, lw=1.8, ls="-", alpha=0.9, zorder=20)
        ax.plot(truth[j], truth[i], marker="o", ms=5.5, color=truth_color, zorder=25)

corner_fig.text(
    0.98,
    0.98,
    "blue = true value",
    color=truth_color,
    ha="right",
    va="top",
    fontsize=11,
)

corner_path = paths.usage_output_dir / f"mcmc_corner_sample_{sample_id:04d}.png"
corner_fig.savefig(corner_path, bbox_inches="tight")
print(f"Saved {corner_path}")
corner_fig

## Fit Overlay on the Selected Data Vector

In [ ]:
median_theta = summary["median"].to_numpy(float)
fit_vector = predict_selected(median_theta)
residual = (fit_vector - data_vector) / np.maximum(np.abs(data_vector), 1.0e-30)

fig, axes = plt.subplots(1, 2, figsize=(11.5, 4.0), constrained_layout=True)
axes[0].plot(data_vector, color="black", lw=1.4, label=f"test sample {sample_id:04d}")
axes[0].plot(fit_vector, color="#d62728", ls="--", lw=1.1, label="MCMC median")
axes[0].set_yscale("symlog", linthresh=1.0e-12)
axes[0].set_xlabel("selected data-vector index")
axes[0].set_ylabel(r"$\zeta_m$")
axes[0].legend(frameon=False, loc="best")

axes[1].axhline(0.0, color="black", lw=0.8)
axes[1].plot(residual, lw=1.1)
axes[1].set_xlabel("selected data-vector index")
axes[1].set_ylabel("(fit - data) / |data|")

fig.suptitle(
    ", ".join(f"{name}={value:.4g}" for name, value in zip(emulator.param_names, median_theta)),
    y=1.03,
)
overlay_path = paths.usage_output_dir / f"mcmc_overlay_sample_{sample_id:04d}.png"
fig.savefig(overlay_path, bbox_inches="tight")
print(f"Saved {overlay_path}")

## Files Written

In [ ]:
for item in sorted(paths.usage_output_dir.glob(f"*sample_{sample_id:04d}*")):
    print(item)